# SCAM (R) vs pyGAM (Python) — single-spec PfPR fit comparison

Fits **one** malaria PfPR model-selection spec in Python (pyGAM) so it can be lined up against
the R `scam` fit produced by `select_malaria_models_rocket.r`. Same parquet, same load/clean
steps, same response (`logit_malaria_pfpr`), no country fixed effect (the stage-1 *screen* form).

**The spec being compared:**

```
logit_malaria_pfpr ~ s(mal_DAH_total_per_capita, k=6, bs="mpd")
                   + s(gdppc_mean, k=6, bs="mpd")
                   + s(weighted_100m_urban_threshold_1500.0_simple_mean, k=6, bs="mpd")
                   + s(precipitation_days, k=6, bs="cv")
                   + s(relative_humidity, k=6, bs="mpi")
                   + malaria_suitability_mordecai_0_0          # linear
```

**scam `bs` → pyGAM constraint mapping**

| scam `bs` | shape | pyGAM |
|-----------|-------|-------|
| `mpd` | monotone decreasing | `s(i, n_splines=6, constraints='monotonic_dec')` |
| `mpi` | monotone increasing | `constraints='monotonic_inc'` |
| `cv`  | concave             | `constraints='concave'` |
| linear| —                   | `l(i)` |

**Caveats — why exact agreement is NOT expected (this is the point of the comparison):**
- scam uses penalized *shape-constrained P-splines*; pyGAM uses penalized B-splines + a constraint penalty. Different bases.
- scam `k=6` (basis dimension) and pyGAM `n_splines=6` are *analogous*, not identical.
- Smoothing-parameter selection differs: scam screen uses **EFS** (per-term `sp`); pyGAM here uses default `lam` and an optional **shared-`lam` gridsearch** (per-term gridsearch is 11⁶ fits — infeasible).
- `AIC` conventions differ (edf accounting). Compare on `pfpr_r`, `dev_expl`/`r_sq`, RMSE/MAE, and the partial-dependence curves — not raw AIC.


In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pygam import LinearGAM, s, l

PARQUET = ("/mnt/team/idd/pub/forecast-mbp/03-modeling_data/malaria/"
           "past_inputs_nc/lsae_1285/current/malaria_past_inputs.parquet")
RESPONSE      = "logit_malaria_pfpr"
SUIT_VARIANT  = "mordecai_0_0"
K             = 6          # scam k  ->  pyGAM n_splines

# (column, form). form in {linear, mpd, mpi, cv, cx}. Order defines the X-matrix columns.
SPEC = [
    ("mal_DAH_total_per_capita",                        "mpd"),
    ("gdppc_mean",                                      "mpd"),
    ("weighted_100m_urban_threshold_1500.0_simple_mean","mpd"),
    ("precipitation_days",                              "cv"),
    ("relative_humidity",                               "mpi"),
    ("malaria_suitability_mordecai_0_0",                "linear"),
]

## 1. Load + clean — mirrors `load_past_data()` in the rocket, verbatim

In [2]:
def load_past_data(parquet_path, suit_variant=SUIT_VARIANT):
    df = pd.read_parquet(parquet_path)
    df["A0_af"] = df["A0_location_id"].astype("category")   # built but the screen drops it (no FE)

    # rocket drops NA on exactly these 3 base columns first
    for v in ["malaria_pfpr", "gdppc_mean", "mal_DAH_total_per_capita"]:
        df = df[df[v].notna()]

    def logit_frac(x, denom):
        f = np.clip(x / denom, 0.001, 0.999)
        return np.log(f / (1 - f))

    df["logit_malaria_suitability"] = logit_frac(df[f"malaria_suitability_{suit_variant}"], 365)
    df["logit_do30"]                = logit_frac(df["days_over_30C"], 365)
    df["logit_relative_humidity"]   = logit_frac(df["relative_humidity"], 100)

    for cov in ["mal_DAH_total_per_capita", "gdppc_mean", "ldipc_mean", "med_consumppc"]:
        df[f"log_{cov}"] = np.log(df[cov])
    return df

past = load_past_data(PARQUET)
print(f"after load_past_data: {len(past):,} rows")

after load_past_data: 313,122 rows


/ihme/homes/bcreiner/miniconda/envs/idd-forecast-mbp/lib/python3.12/site-packages/pandas/core/arraylike.py:399: RuntimeWarning: divide by zero encountered in log
  result = getattr(ufunc, method)(*inputs, **kwargs)


## 2. Model frame — replicate scam's `na.action = na.omit`

scam drops any row with NA in *any* variable used by **this** spec. We drop NA on the spec
columns + response (+ keep `malaria_pfpr` for the response-scale metric, and `location_id`/
`year_id` as keys so the R fit can be aligned row-for-row).

In [3]:
feature_cols = [v for v, _ in SPEC]
key_cols     = [c for c in ["location_id", "year_id"] if c in past.columns]
model_cols   = feature_cols + [RESPONSE, "malaria_pfpr"]

frame = past.dropna(subset=model_cols).copy()
frame = frame.sort_values(key_cols).reset_index(drop=True) if key_cols else frame.reset_index(drop=True)

X       = frame[feature_cols].to_numpy(dtype="float64")
y       = frame[RESPONSE].to_numpy(dtype="float64")
pfpr_ac = frame["malaria_pfpr"].to_numpy(dtype="float64")
print(f"n_obs used in fit: {len(frame):,}   (report this to the R side — it must match scam's n_obs)")
print("feature column order:", feature_cols)

n_obs used in fit: 313,122   (report this to the R side — it must match scam's n_obs)
feature column order: ['mal_DAH_total_per_capita', 'gdppc_mean', 'weighted_100m_urban_threshold_1500.0_simple_mean', 'precipitation_days', 'relative_humidity', 'malaria_suitability_mordecai_0_0']


## 3. Build the pyGAM term structure from the spec and fit

In [4]:
CONSTRAINT = {"mpd": "monotonic_dec", "mpi": "monotonic_inc", "cv": "concave", "cx": "convex"}

def build_terms(spec, n_splines=K):
    terms = None
    for i, (_, form) in enumerate(spec):
        term = l(i) if form == "linear" else s(i, n_splines=n_splines, constraints=CONSTRAINT[form])
        terms = term if terms is None else terms + term
    return terms

gam = LinearGAM(build_terms(SPEC))          # Gaussian + identity == scam default family
gam.fit(X, y)
print(gam.summary())

LinearGAM                                                                                                 
=============================================== ==========================================================
Distribution:                        NormalDist Effective DoF:                                      7.9067
Link Function:                     IdentityLink Log Likelihood:                               -949894.7751
Number of Samples:                       313122 AIC:                                          1899807.3637
                                                AICc:                                         1899807.3643
                                                GCV:                                               25.2656
                                                Scale:                                              5.0264
                                                Pseudo R-Squared:                                    0.428
Feature Function                  Lam

/tmp/ipykernel_378342/3859511536.py:12: UserWarning: KNOWN BUG: p-values computed in this summary are likely much smaller than they should be. 
 
Please do not make inferences based on these values! 

Collaborate on a solution, and stay up to date at: 
github.com/dswah/pyGAM/issues/163 

  print(gam.summary())


## 4. Metrics — computed from `y` and predictions with the *exact* formulas in `is_block_metrics`

These mirror the rocket's `is_no_fe_*` columns so they can be put next to the R parquet row.
`pfpr_r` (Pearson r between observed prevalence and `expit(fitted)`) is the metric the screen
ranks on — it's the headline number.

In [5]:
def expit(z): return 1.0 / (1.0 + np.exp(-z))

def metrics(gam, X, y_logit, pfpr_actual, label):
    pred   = gam.predict(X)                       # logit scale
    resid  = y_logit - pred
    sse    = float(np.sum(resid**2))
    sst    = float(np.sum((y_logit - y_logit.mean())**2))
    pfpr_p = expit(pred)
    return {
        "model":        label,
        "n_obs":        int(len(y_logit)),
        "edof":         float(gam.statistics_["edof"]),
        "deviance":     sse,                       # gaussian deviance == SSE
        "null_deviance":sst,
        "dev_expl":     1 - sse/sst,
        "r_sq":         1 - sse/sst,               # == dev_expl for gaussian identity
        "rmse":         float(np.sqrt(np.mean(resid**2))),
        "mae":          float(np.mean(np.abs(resid))),
        "pfpr_rmse":    float(np.sqrt(np.mean((pfpr_actual - pfpr_p)**2))),
        "pfpr_mae":     float(np.mean(np.abs(pfpr_actual - pfpr_p))),
        "pfpr_r":       float(np.corrcoef(pfpr_actual, pfpr_p)[0, 1]),
        "pygam_AIC":    float(gam.statistics_["AIC"]),   # NOT comparable to scam AIC (edf convention)
    }

rows = [metrics(gam, X, y, pfpr_ac, "pyGAM default lam")]
pd.DataFrame(rows).set_index("model").T

model,pyGAM default lam
n_obs,3.131220e+05
edof,7.906745e+00
deviance,7.910657e+06
null_deviance,1.382924e+07
dev_expl,4.279759e-01
r_sq,4.279759e-01
rmse,5.026313e+00
mae,4.062747e+00
pfpr_rmse,1.732151e-01
pfpr_mae,7.738404e-02


## 5. (Optional) shared-`lam` gridsearch

The closest tractable analog to scam's EFS smoothing-param selection. Sweeps a *single* `lam`
across all terms (per-term search is infeasible). ~1 min on this row count — skip if not needed.

In [6]:
RUN_GRIDSEARCH = True
if RUN_GRIDSEARCH:
    gam_gs = LinearGAM(build_terms(SPEC)).gridsearch(
        X, y, lam=np.logspace(-3, 3, 11), progress=False)
    rows.append(metrics(gam_gs, X, y, pfpr_ac, "pyGAM gridsearch lam"))
    print("best lam (per term):", gam_gs.lam)
pd.DataFrame(rows).set_index("model").T

best lam (per term): [[np.float64(0.003981071705534973)], [np.float64(0.003981071705534973)], [np.float64(0.003981071705534973)], [np.float64(0.003981071705534973)], [np.float64(0.003981071705534973)], [np.float64(0.003981071705534973)]]


model,pyGAM default lam,pyGAM gridsearch lam
n_obs,3.131220e+05,3.131220e+05
edof,7.906745e+00,7.960622e+00
deviance,7.910657e+06,7.910250e+06
null_deviance,1.382924e+07,1.382924e+07
dev_expl,4.279759e-01,4.280054e-01
r_sq,4.279759e-01,4.280054e-01
rmse,5.026313e+00,5.026183e+00
mae,4.062747e+00,4.062343e+00
pfpr_rmse,1.732151e-01,1.731883e-01
pfpr_mae,7.738404e-02,7.737923e-02


## 6. Partial-dependence curves (the 5 smooths) + the linear term

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
for i, (col, form) in enumerate(SPEC):
    ax = axes.flat[i]
    XX = gam.generate_X_grid(term=i)
    pdep, ci = gam.partial_dependence(term=i, X=XX, width=0.95)
    ax.plot(XX[:, i], pdep)
    ax.plot(XX[:, i], ci, ls="--", c="grey")
    ax.set_title(f"{col}\n[{form}]", fontsize=9)
    ax.set_xlabel(col, fontsize=7)
for j in range(len(SPEC), len(axes.flat)):
    axes.flat[j].axis("off")
fig.suptitle("pyGAM partial dependence — compare shapes against scam plot.gam", y=1.02)
plt.tight_layout(); plt.show()

## 7. (Optional) dump per-row fitted values for a row-by-row R↔Python diff

Both sides load the same parquet and drop NA on the same spec columns, so row order should match
on `(location_id, year_id)`. Uncomment to write a keyed parquet the R CC can join against.
*(Writes into the repo output area, not /tmp — adjust the path before running.)*

In [ ]:
# out = frame[key_cols].copy()
# out["pred_logit_pygam"] = gam.predict(X)
# out["pred_pfpr_pygam"]  = expit(out["pred_logit_pygam"])
# out.to_parquet("pygam_fitted_<spec_label>.parquet", index=False)
# print("wrote", len(out), "rows")